<a href="https://colab.research.google.com/github/chaitra0312/ML/blob/main/clashug.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets scikit-learn torch


In [ ]:
from transformers import pipeline

# Load the fine-tuned BART summarizer
summarizer = pipeline("summarization", model="kochi-metro24/bart-metro-summarizer")

# Test the summarizer
doc_text = """
Since its first commercial run in 2017, KMRL has grown into a complex, multidisciplinary enterprise
that stretches far beyond train operations...
"""
summary = summarizer(doc_text, max_length=50, min_length=10, do_sample=False)[0]['summary_text']
print("Summary:", summary)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/302 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/957 [00:00<?, ?B/s]

Device set to use cuda:0
Your max_length is set to 50, but your input_length is only 32. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=16)
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Summary: Since its first commercial run in 2017, KMRL has grown into a complex, multidisciplinary enterprise—that stretches far beyond train operations...


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Load the 80-example CSV
df = pd.read_csv("/content/kmrl_classifier_dataset_500.csv")

# Train-test split (demo)
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['label'], test_size=0.2, random_state=42)


In [ ]:
# Create pipeline
classifier_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression(max_iter=200))
])

# Train classifier
classifier_pipeline.fit(X_train, y_train)

# Test accuracy (optional)
accuracy = classifier_pipeline.score(X_test, y_test)
print(f"Classifier Accuracy: {accuracy*100:.2f}%")


Classifier Accuracy: 96.00%


In [ ]:
def summarize_and_classify(doc_text):
    # Step 1: Summarize
    summary = summarizer(doc_text, max_length=50, min_length=10, do_sample=False)[0]['summary_text']

    # Step 2: Predict Role
    role = classifier_pipeline.predict([summary])[0]

    return {"summary": summary, "role": role}

# Example
doc = "Emergency drill report indicates delayed evacuation timing in station 5. Fire extinguishers missing in multiple areas."
result = summarize_and_classify(doc)
print(result)


Your max_length is set to 50, but your input_length is only 21. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=10)
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'summary': 'High passenger load and delays: delayed evacuation in station.', 'role': 'Safety Officer'}


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import joblib

# Save trained classifier pipeline
joblib.dump(classifier_pipeline, '/content/drive/MyDrive/kmrl_classifier_pipeline_500.joblib')


['/content/drive/MyDrive/kmrl_classifier_pipeline_500.joblib']

In [ ]:
from transformers import BartForConditionalGeneration, BartTokenizer

bart_model = BartForConditionalGeneration.from_pretrained("kochi-metro24/bart-metro-summarizer")
bart_tokenizer = BartTokenizer.from_pretrained("kochi-metro24/bart-metro-summarizer")


In [ ]:
bart_model.save_pretrained('/content/drive/MyDrive/kmrl_bart_model')
bart_tokenizer.save_pretrained('/content/drive/MyDrive/kmrl_bart_tokenizer')


('/content/drive/MyDrive/kmrl_bart_tokenizer/tokenizer_config.json',
 '/content/drive/MyDrive/kmrl_bart_tokenizer/special_tokens_map.json',
 '/content/drive/MyDrive/kmrl_bart_tokenizer/vocab.json',
 '/content/drive/MyDrive/kmrl_bart_tokenizer/merges.txt',
 '/content/drive/MyDrive/kmrl_bart_tokenizer/added_tokens.json')